# Pokrivenost naspram granularnosti na gustim grafovima (E1)

Ponavlja eksperiment razdvajanja pokrivenosti od granularnosti particionisanja na grafu
koji se bira parametrom GRAF (Reddit ili ogbn-products). Dve grane pri istom K:

| Grana | Klastera po batch-u | Pokrivenost po koraku |
|---|---|---|
| fiksno | 10 za sve K | opada sa K (5 do 0,5 odsto) |
| pokrivenost | K puta 0,05 | konstantnih 5 odsto |

Pri K = 200 su grane identicne (10 klastera = 5 odsto), pa se ta konfiguracija izvrsava
jednom i sluzi obema granama. Ukupno 7 konfiguracija po seedu, 70 treninga.

Pokrivenost je 5 odsto (na arxiv-u je bila 20) zbog memorije, batch od 20 odsto Reddit-a
sa 602 atributa ne staje u karticu pri razmeni poruka. Protokol: GraphSAGE, L = 3,
skriveni 256, lr 0,01, 2.500 gradijentnih koraka, izbor najbolje kontrolne tacke
uzorkovanom ocenom na validaciji na svakih 250 koraka (fanout [15, 10, 5], fiksiran seed
po oceni, identican za sve konfiguracije), a konacna tacnost se racuna tacnim
zakljucivanjem po slojevima sa svim susedima, jednom po treningu, za najbolju tacku.
Ocena nad celim grafom u jednom prolazu ovde nije moguca, jer bi razmena poruka
materijalizovala atribute po potegu (Reddit preko 200 GB).

Meri se i energija kartice NVML brojacem (pokriva trening i medjuocene, ne zavrsno
zakljucivanje) i trenutna snaga po koraku.

Procena trajanja: Reddit oko 8 do 10 sati, products oko 10 do 14 sati. Merenje preskace
vec upisane redove, pa se moze podeliti na vise sesija po granici seeda. Tacnost i
pokrivenost su hardverski nezavisne, a vremena porediti samo unutar iste sesije.


In [ ]:
GRAF = 'Reddit'
SEEDOVI = [42, 123, 7, 1, 2, 3, 4, 5, 6, 8]
KLASTERI = [200, 500, 1000, 2000]
FIKSNO = 10
POKRIVENOST = 0.05
KORAKA, SLOJEVA, OCENA_SVAKIH = 2500, 3, 250
EVAL_FANOUT = [15, 10, 5]
INFER_BATCH = 2048 if GRAF == 'Reddit' else 4096


In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging, copy
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import ClusterData, ClusterLoader, NeighborLoader
from torch_geometric.utils import to_undirected, subgraph
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}')

REZ = Path('rezultati_pokrivenost_gustina/' + GRAF.replace('ogbn-', ''))
REZ.mkdir(parents=True, exist_ok=True)

try:
    import pynvml
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-ml-py'], check=False)
    import pynvml
try:
    pynvml.nvmlInit()
    _ruka = pynvml.nvmlDeviceGetHandleByIndex(0)
    log.info(f'NVML aktivan, trenutna snaga {pynvml.nvmlDeviceGetPowerUsage(_ruka)/1000:.0f} W')
except Exception:
    _ruka = None
    log.info('NVML nedostupan, energija se ne meri')


def energija_j():
    if _ruka is None:
        return float('nan')
    try:
        return pynvml.nvmlDeviceGetTotalEnergyConsumption(_ruka) / 1000.0
    except pynvml.NVMLError:
        return float('nan')


def snaga_w():
    if _ruka is None:
        return float('nan')
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_ruka) / 1000.0
    except pynvml.NVMLError:
        return float('nan')


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = REZ / f'run_{seed}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(putanja, red):
    novi = not putanja.exists()
    with open(putanja, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(putanja, kolona):
    if not putanja.exists():
        return set()
    with open(putanja) as f:
        return {r[kolona] for r in csv.DictReader(f)}


## Model i korak treninga

In [ ]:
SKRIVENI, LR = 256, 0.01


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    maska = batch.train_mask
    if maska.sum() == 0:
        return
    gubitak = F.cross_entropy(out[maska], batch.y[maska])
    gubitak.backward()
    opt.step()


## Ucitavanje grafa

In [ ]:
if GRAF == 'Reddit':
    from torch_geometric.datasets import Reddit
    data = Reddit(root='podaci/Reddit')[0]
    broj_klasa = 41
    val_idx = data.val_mask.nonzero(as_tuple=True)[0]
    test_idx = data.test_mask.nonzero(as_tuple=True)[0]
else:
    dataset = PygNodePropPredDataset(name=GRAF, root='podaci')
    data = dataset[0]
    data.y = data.y.squeeze()
    broj_klasa = dataset.num_classes
    split_idx = dataset.get_idx_split()
    data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    data.train_mask[split_idx['train']] = True
    val_idx, test_idx = split_idx['valid'], split_idx['test']
log.info(f'{GRAF}: {data.num_nodes:,} cvorova, {data.num_edges:,} potega, {broj_klasa} klasa')


## Ocena, uzorkovana za izbor i tacna za konacan rezultat

In [ ]:
@torch.no_grad()
def oceni_uzorkovano(model, seed, korak):
    model.eval()
    torch.manual_seed(seed * 1000003 + korak)
    loader = NeighborLoader(data, num_neighbors=EVAL_FANOUT, input_nodes=val_idx,
                            batch_size=8192, shuffle=False, num_workers=0)
    tacnih, ukupno = 0, 0
    for b in loader:
        b = b.to(device, non_blocking=True)
        out = model(b.x, b.edge_index)[:b.batch_size]
        tacnih += int((out.argmax(-1) == b.y[:b.batch_size]).sum())
        ukupno += b.batch_size
    return tacnih / ukupno


@torch.no_grad()
def zakljuci_tacno(model):
    model.eval()
    loader = NeighborLoader(data, num_neighbors=[-1], batch_size=INFER_BATCH,
                            shuffle=False, num_workers=8, persistent_workers=False)
    x_sve = data.x
    for i in range(model.slojeva):
        delovi = []
        for b in loader:
            x = x_sve[b.n_id].to(device, non_blocking=True)
            ei = b.edge_index.to(device, non_blocking=True)
            h = model.convs[i](x, ei)[:b.batch_size]
            if i < model.slojeva - 1:
                h = F.relu(model.bns[i](h))
            delovi.append(h.cpu())
            del x, ei, h
        x_sve = torch.cat(delovi)
        torch.cuda.empty_cache()
    pred = x_sve.argmax(-1)
    val = float((pred[val_idx] == data.y[val_idx]).float().mean())
    test = float((pred[test_idx] == data.y[test_idx]).float().mean())
    return val, test


## Proba ocene pre merenja

In [ ]:
proba = GraphSAGE(data.num_features, SKRIVENI, broj_klasa, SLOJEVA).to(device)
t0 = time.time()
v = oceni_uzorkovano(proba, 0, 0)
log.info(f'proba uzorkovane ocene: {v*100:.1f} % (neobucen model), {time.time()-t0:.1f} s')
t0 = time.time()
pv, pt = zakljuci_tacno(proba)
log.info(f'proba tacnog zakljucivanja: val {pv*100:.1f} %, test {pt*100:.1f} %, {time.time()-t0:.0f} s')
del proba
gc.collect()
torch.cuda.empty_cache()


## Merenje

In [ ]:
KONFIG = [('fiksno', K, FIKSNO) for K in KLASTERI]
KONFIG += [('pokrivenost', K, round(K * POKRIVENOST)) for K in KLASTERI
           if round(K * POKRIVENOST) != FIKSNO]

particije = {}
for K in KLASTERI:
    t0 = time.time()
    particije[K] = (ClusterData(data, num_parts=K, recursive=False), time.time() - t0)
    log.info(f'K={K}: METIS {particije[K][1]:.0f} s')

for SEED in SEEDOVI:
    fajl = direktorijum(SEED) / 'pokrivenost.csv'
    uradjeno = vec_uradjeno(fajl, 'oznaka')
    for grana, K, batch_klastera in KONFIG:
        oznaka = f'{grana}_K{K}'
        if oznaka in uradjeno:
            continue
        cd, metis_vreme = particije[K]
        postavi_seed(SEED)
        loader = ClusterLoader(cd, batch_size=batch_klastera, shuffle=True,
                               num_workers=8, persistent_workers=True)
        model = GraphSAGE(data.num_features, SKRIVENI, broj_klasa, SLOJEVA).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_korak, najbolji = 0.0, 0, None
        vrh, trening_vreme, korak, epoha = 0.0, 0.0, 0, 0
        cvorova_l, potega_l, snage = [], [], []
        e0 = energija_j()
        torch.cuda.reset_peak_memory_stats()
        while korak < KORAKA:
            epoha += 1
            model.train()
            for batch in loader:
                torch.cuda.synchronize()
                t0 = time.time()
                trening_korak(model, batch, opt)
                torch.cuda.synchronize()
                vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
                trening_vreme += time.time() - t0
                snage.append(snaga_w())
                if len(cvorova_l) < 20:
                    cvorova_l.append(batch.num_nodes)
                    potega_l.append(batch.num_edges)
                korak += 1
                if korak % OCENA_SVAKIH == 0:
                    v = oceni_uzorkovano(model, SEED, korak)
                    if v > naj_val:
                        naj_val, naj_korak = v, korak
                        najbolji = {k: t.cpu().clone() for k, t in model.state_dict().items()}
                    model.train()
                    torch.cuda.reset_peak_memory_stats()
                if korak >= KORAKA:
                    break
        e1 = energija_j()

        model.load_state_dict(najbolji)
        val_tacno, test_tacno = zakljuci_tacno(model)
        upisi_red(fajl, {
            'oznaka': oznaka, 'grana': grana, 'klastera': K,
            'batch_klastera': batch_klastera,
            'pokrivenost_pct': batch_klastera / K * 100,
            'ukupno_koraka': korak, 'epoha': epoha, 'metis_s': metis_vreme,
            'naj_val_uzorak': naj_val, 'naj_korak': naj_korak,
            'val_tacno': val_tacno, 'test_tacno': test_tacno,
            'cvorova_po_koraku': float(np.mean(cvorova_l)),
            'potega_po_koraku': float(np.mean(potega_l)),
            'vreme_po_koraku_ms': trening_vreme / korak * 1000,
            'memorija_mb': vrh, 'energija_j': e1 - e0,
            'snaga_prosek_w': float(np.nanmean(snage)),
            'snaga_min_w': float(np.nanmin(snage)),
            'snaga_max_w': float(np.nanmax(snage))})
        log.info(f'seed {SEED} | {oznaka}: test={test_tacno*100:.2f}% (val {val_tacno*100:.2f}, '
                 f'korak {naj_korak}), {trening_vreme/korak*1000:.0f} ms/korak, vrh {vrh:.0f} MB, '
                 f'{e1-e0:.0f} J')
        del model, opt, loader, najbolji
        gc.collect()
        torch.cuda.empty_cache()

log.info('Merenje zavrseno za sve seedove')


## Sazetak

In [ ]:
for grana, K, bk in KONFIG:
    t, m, e = [], [], []
    for SEED in SEEDOVI:
        p = direktorijum(SEED) / 'pokrivenost.csv'
        if not p.exists():
            continue
        with open(p) as f:
            redovi = {r['oznaka']: r for r in csv.DictReader(f)}
        r = redovi.get(f'{grana}_K{K}')
        if r is None:
            continue
        t.append(float(r['test_tacno']) * 100)
        m.append(float(r['memorija_mb']))
        e.append(float(r['energija_j']))
    if len(t) > 1:
        log.info(f'{grana:12s} K={K:<5d} ({bk} kl/batch): test {np.mean(t):.2f} +/- {np.std(t, ddof=1):.2f} %, '
                 f'vrh {np.mean(m):.0f} MB, energija {np.mean(e):.0f} J')


## Arhiviranje

In [ ]:
import zipfile

arhiva = Path('rezultati_pokrivenost_gustina.zip')
with zipfile.ZipFile(arhiva, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REZ.rglob('*')):
        if p.is_file():
            z.write(p)
log.info(f'{arhiva}: {arhiva.stat().st_size / 1e6:.1f} MB, preuzeti i sacuvati u projekat')
